In [12]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
BATCH_SIZE = 128
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3


In [14]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2470, 0.2435, 0.2616],
    ),
])


In [15]:
train_dataset = CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

test_dataset = CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
)

classes = train_dataset.classes
num_classes = len(classes)

print("Device:", DEVICE)
print("Classes:", classes)


  0%|          | 0/170498071 [00:00<?, ?it/s]

Extracting data\cifar-10-python.tar.gz to data
Files already downloaded and verified
Device: cpu
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [17]:

class Vectorizer(nn.Module):
    """
    Превращает изображение 3x32x32 в вектор признаков размерности 256.
    """
    def __init__(self, embedding_dim=256):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )

        self.flatten = nn.Flatten()
        self.projection = nn.Linear(128, embedding_dim)

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.projection(x)
        return x


In [18]:
class Classifier(nn.Module):
    """
    Классифицирует вектор признаков.
    """
    def __init__(self, embedding_dim=256, num_classes=10):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)

In [19]:
class ImageClassifier(nn.Module):
    def __init__(self, embedding_dim=256, num_classes=10):
        super().__init__()

        self.vectorizer = Vectorizer(embedding_dim)
        self.classifier = Classifier(embedding_dim, num_classes)

    def forward(self, x):
        embedding = self.vectorizer(x)
        logits = self.classifier(embedding)
        return logits, embedding

In [20]:
model = ImageClassifier(
    embedding_dim=256,
    num_classes=num_classes,
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [21]:
def train_one_epoch():
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, targets in train_loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        optimizer.zero_grad()

        logits, _ = model(images)
        loss = criterion(logits, targets)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == targets).sum().item()
        total += targets.size(0)

    return total_loss / total, correct / total

In [22]:
@torch.no_grad()
def evaluate():
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, targets in test_loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        logits, _ = model(images)
        loss = criterion(logits, targets)

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == targets).sum().item()
        total += targets.size(0)

    return total_loss / total, correct / total


for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch()
    test_loss, test_acc = evaluate()

    print(
        f"Epoch {epoch:02d} | "
        f"train loss: {train_loss:.4f}, acc: {train_acc:.4f} | "
        f"test loss: {test_loss:.4f}, acc: {test_acc:.4f}"
    )

torch.save(model.state_dict(), "cifar10_vector_classifier.pth")
print("Saved: cifar10_vector_classifier.pth")

Epoch 01 | train loss: 1.7873, acc: 0.3092 | test loss: 1.5507, acc: 0.4065
Epoch 02 | train loss: 1.4567, acc: 0.4482 | test loss: 1.3322, acc: 0.5093
Epoch 03 | train loss: 1.2708, acc: 0.5298 | test loss: 1.1940, acc: 0.5619
Epoch 04 | train loss: 1.1400, acc: 0.5823 | test loss: 1.0983, acc: 0.5966
Epoch 05 | train loss: 1.0336, acc: 0.6254 | test loss: 1.0220, acc: 0.6253
Saved: cifar10_vector_classifier.pth


In [ ]:
# ResNet


In [5]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# 1. Загружаем предобученную модель ResNet-18
#weights = models.ResNet18_Weights.DEFAULT
#model = models.resnet18(weights=weights)

model = models.resnet18(pretrained=True)

# 2. Убираем последний полносвязный слой (fc), чтобы на выходе получался вектор 512
# Способ через nn.Sequential (отрезаем всё после avgpool/flatten)
feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])
model.eval()  # Переводим в режим инференса

# 3. Подготавливаем стандартные трансформации для ImageNet
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 4. Загружаем и предобрабатываем изображение
img_path = 'datasets/cat1.jpg'  
image = Image.open(img_path).convert('RGB')
input_tensor = transform(image).unsqueeze(0)  # Добавляем размерность батча: [1, 3, 224, 224]

# 5. Извлекаем вектор признаков
with torch.no_grad():
    feature_vector = feature_extractor(input_tensor)

# Убираем лишние размерности (из [1, 512, 1, 1] получаем вектор [512])
feature_vector = torch.flatten(feature_vector, 1)

print("Размерность вектора:", feature_vector.shape)  # torch.Size([1, 512])
print("Вектор:", feature_vector[0][:10])  # Первые 10 чисел эмбеддинга


v1 = feature_vector


img_path = 'datasets/cat2.jpg'  
image = Image.open(img_path).convert('RGB')
input_tensor = transform(image).unsqueeze(0)  # Добавляем размерность батча: [1, 3, 224, 224]

with torch.no_grad():
    feature_vector = feature_extractor(input_tensor)

v2 = torch.flatten(feature_vector, 1)



img_path = 'datasets/dog1.jpg'  
image = Image.open(img_path).convert('RGB')
input_tensor = transform(image).unsqueeze(0)  # Добавляем размерность батча: [1, 3, 224, 224]

with torch.no_grad():
    feature_vector = feature_extractor(input_tensor)

v3 = torch.flatten(feature_vector, 1)


Размерность вектора: torch.Size([1, 512])
Вектор: tensor([0.0000, 0.4358, 1.4920, 0.4389, 0.1054, 0.2209, 0.0591, 0.3533, 0.6251,
        0.0822])


In [7]:
# 1. Принудительно превращаем векторы в одномерные массивы [512]
vec1_flat = v1.squeeze()
vec2_flat = v2.squeeze()
vec3_flat = v3.squeeze()

In [8]:
# Расчет косинусной близости (требует размерности как минимум 2D, поэтому добавляем dim=0)
similarity = torch.nn.functional.cosine_similarity(vec1_flat, vec2_flat, dim=0)
print(f"Косинусная близость кот 1 и кот 2: {similarity.item():.4f}")

similarity = torch.nn.functional.cosine_similarity(vec2_flat, vec3_flat, dim=0)
print(f"Косинусная близость кот и собака: {similarity.item():.4f}")



Косинусная близость кот 1 и кот 2: 0.8487
Косинусная близость кот и собака: 0.5529
